In [ ]:

using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120
const aa = 40
const N = 1_000_000
const I0 = 10
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)

data_file_for(tag::Symbol) =
    tag === :memoryless  ? "data/memoryless.csv" :
    tag === :sliding     ? "data/sliding_kmax14.csv" :          
    tag === :powerlaw    ? "data/powerlaw_lambdaP.csv" :
    tag === :exponential ? "data/exponential_lambdaE.csv" :
    tag === :reciprocal  ? "data/reciprocal_lambdaR.csv" :     
    error("Unknown data tag: $tag")

model_tag_sym = :exponential
data_tag_sym  = :exponential

KMAX_UPPER = 30  

data_path = data_file_for(data_tag_sym)
raw, hdr = DelimitedFiles.readdlm(data_path, ',', header=true)
raw = Matrix{Float64}(raw)
tau = size(raw, 2) ÷ 3

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

i = 15
c = 3
Istar_obs = Vector{Int}(round.(Int, raw[i, 1:tau]))
@info "[$(String(model_tag_sym))_model_$(String(data_tag_sym))_data] Fitting dataset $(i) chain $(c) (tau=$tau)"

Random.seed!(2025 + i * 100 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_sim_$(i)_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_sim_$(i)_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))

    el = Dates.value(Dates.now() - t0) / 1000
    @info(@sprintf("Dataset %03d chain %d: ok in %.2fs", i, c, el))
catch err
    el = Dates.value(Dates.now() - t0) / 1000
    @warn(@sprintf("Dataset %03d chain %d: ERROR - %s after %.2fs", i, c, err))
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [exponential_model_exponential_data] Fitting dataset 15 chain 3 (tau=50)
[ Info: [exponential] iter 1000/1000000 elapsed=3.4s, rate=0.199, medians=[0.750, 0.00464, 0.295, 0.320], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAPT]
[ Info: [exponential] iter 2000/1000000 elapsed=7.4s, rate=0.206, medians=[0.760, 0.00455, 0.291, 0.317], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAPT]
[ Info: [exponential] iter 3000/1000000 elapsed=10.5s, rate=0.192, medians=[0.760, 0.00459, 0.285, 0.308], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAPT]
[ Info: [exponential] iter 4000/1000000 elapsed=13.7s, rate=0.184, medians=[0.762, 0.00455, 0.280, 0.304], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAPT]
[ Info: [exponential] iter 5000/1000000 elapsed=16.8s, rate=0.177, medians=[0.768, 0.00448, 0.278, 0.305], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAPT]
[ Info: [exponential] iter 6000/1000000 elapsed=20.0s, rate=0.172, medians=[0.772, 0.00444, 0.276, 0.305], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAP